# SAM 3.1 building footprints: Colab run

Zero-shot building segmentation with SAM 3.1 on the site's held-out Puerto Rico tiles. Produces the files for `assets/geoai/`.

**Before running**
1. *Runtime → Change runtime type → T4 GPU* (any GPU works).
2. Your Hugging Face account needs access to the gated [facebook/sam3.1](https://huggingface.co/facebook/sam3.1) repo.
3. Add your HF token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and allow notebook access.

Then *Runtime → Run all*. The last cell downloads `geoai_assets.zip`; unzip it over `assets/geoai/` in the site repo.

The prompt is chosen on **Bayamón**, a training-set area, so the Ponce/Guaynabo scores are never used to tune anything.

In [ ]:
!pip -q install -U "transformers>=5.5" huggingface_hub opencv-python-headless
import torch; print(torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login()   # falls back to a paste-in prompt

In [ ]:
%mkdir -p /content/geoai
%cd /content/geoai

In [ ]:
%%writefile acquire.py
import math, json, os, subprocess, time, sys
import numpy as np
from PIL import Image, ImageDraw

R = 6378137.0
def ll2m(lat, lon):
    return R*math.radians(lon), R*math.log(math.tan(math.pi/4 + math.radians(lat)/2))

AOIS = [
    ("sanjuan",   18.4505, -66.0630),  # dense urban Santurce
    ("bayamon",   18.3989, -66.1614),  # suburban
    ("caguas",    18.2341, -66.0356),
    ("ponce",     18.0111, -66.6141),
    ("mayaguez",  18.2013, -67.1397),
    ("carolina",  18.3950, -65.9600),
    ("arecibo",   18.4655, -66.7285),
    ("guaynabo",  18.3580, -66.1110),
]
# optional AOI names on the command line limit the fetch, e.g. `acquire.py ponce guaynabo`
AOIS = [a for a in AOIS if not sys.argv[1:] or a[0] in sys.argv[1:]]
HALF = 600        # metres -> 1200 m box
PX   = 1536       # -> 0.78 m/px
os.makedirs("data", exist_ok=True)

def fetch_img(name, bbox):
    out = f"data/{name}_img.jpg"
    if os.path.exists(out) and os.path.getsize(out) > 5000: return out
    url = ("https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/export"
           f"?bbox={bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}&bboxSR=3857&imageSR=3857"
           f"&size={PX},{PX}&format=jpg&f=image")
    subprocess.run(["curl","-s","-m","120",url,"-o",out], check=True)
    return out

def fetch_osm(name, lat, lon, dlat, dlon):
    out = f"data/{name}_osm.json"
    if os.path.exists(out) and os.path.getsize(out) > 100: return out
    q = (f'[out:json][timeout:60];('
         f'way["building"]({lat-dlat},{lon-dlon},{lat+dlat},{lon+dlon});'
         f'relation["building"]({lat-dlat},{lon-dlon},{lat+dlat},{lon+dlon});'
         f');out geom;')
    mirrors=["https://overpass-api.de/api/interpreter",
             "https://overpass.kumi.systems/api/interpreter",
             "https://overpass.osm.jp/api/interpreter"]
    import json as _j
    for attempt in range(6):
        url=mirrors[attempt % len(mirrors)]
        subprocess.run(["curl","-s","-m","180","-A","geoai-portfolio/1.0",
                        "-X","POST",url,
                        "--data-urlencode",f"data={q}","-o",out])
        try:
            _j.load(open(out)); return out
        except Exception:
            wait=8*(attempt+1)
            print(f"    overpass retry {attempt+1} for {name} (wait {wait}s)", flush=True)
            time.sleep(wait)
    raise RuntimeError("overpass failed for "+name)
    return out

def rasterize(name, bbox, osm_path):
    d = json.load(open(osm_path))
    x0,y0,x1,y1 = bbox
    mask = Image.new("L",(PX,PX),0); drw = ImageDraw.Draw(mask)
    n=0
    for el in d.get("elements",[]):
        rings=[]
        if el.get("type")=="way" and el.get("geometry"):
            rings=[el["geometry"]]
        elif el.get("type")=="relation":
            for m in el.get("members",[]):
                if m.get("role")=="outer" and m.get("geometry"): rings.append(m["geometry"])
        for g in rings:
            pts=[]
            for p in g:
                mx,my = ll2m(p["lat"], p["lon"])
                px = (mx-x0)/(x1-x0)*PX
                py = (y1-my)/(y1-y0)*PX
                pts.append((px,py))
            if len(pts)>=3:
                drw.polygon(pts, fill=255); n+=1
    mask.save(f"data/{name}_mask.png")
    return n

meta={}
for name, lat, lon in AOIS:
    cx,cy = ll2m(lat,lon)
    bbox=(cx-HALF, cy-HALF, cx+HALF, cy+HALF)
    dlat = HALF/111320.0 * 1.15
    dlon = HALF/(111320.0*math.cos(math.radians(lat))) * 1.15
    fetch_img(name,bbox)
    fetch_osm(name,lat,lon,dlat,dlon)
    nb = rasterize(name,bbox,f"data/{name}_osm.json")
    im=np.asarray(Image.open(f"data/{name}_img.jpg").convert("RGB"))
    mk=np.asarray(Image.open(f"data/{name}_mask.png"))
    cov = (mk>127).mean()*100
    meta[name]=dict(bbox=bbox, buildings=nb, coverage=round(float(cov),2), lat=lat, lon=lon)
    print(f"  {name:10s} buildings={nb:5d}  mask_cov={cov:5.2f}%  img_std={im.std():.1f}")
    time.sleep(2)
json.dump(meta, open("data/meta.json","w"), indent=1)
print("saved", len(meta), "AOIs")


In [ ]:
%%writefile convert_sam3_to_hf.py
# Copyright 2025 The Meta AI Authors and The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Convert SAM3 checkpoints from the original implementation to HuggingFace format.

Original repository: https://github.com/facebookresearch/segment-anything-3
"""

import argparse
import gc
import os

import regex as re
import torch

from transformers import CLIPTokenizerFast, Sam3Config, Sam3ImageProcessor, Sam3Model, Sam3Processor
from transformers.utils import logging


logging.set_verbosity_info()
logger = logging.get_logger(__name__)

# fmt: off
ORIGINAL_TO_CONVERTED_KEY_MAPPING = {
    r"^sam3_model\.": r"",
    # ============================================================================
    # Vision Encoder - ViT Backbone
    # ============================================================================
    r"^backbone\.vision_backbone\.trunk\.":                                 r"vision_encoder.backbone.",
    r"^vision_encoder\.backbone\.pos_embed":                                r"vision_encoder.backbone.embeddings.position_embeddings",
    r"^vision_encoder\.backbone\.patch_embed\.proj\.":                      r"vision_encoder.backbone.embeddings.patch_embeddings.projection.",
    r"^vision_encoder\.backbone\.ln_pre\.":                                 r"vision_encoder.backbone.layer_norm.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.norm1\.":                   r"vision_encoder.backbone.layers.\1.layer_norm1.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.norm2\.":                   r"vision_encoder.backbone.layers.\1.layer_norm2.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.attn\.qkv\.":               r"vision_encoder.backbone.layers.\1.attention.qkv.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.attn\.proj\.":              r"vision_encoder.backbone.layers.\1.attention.o_proj.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.attn\.freqs_cis":           r"vision_encoder.backbone.layers.\1.rotary_emb.rope_embeddings",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.mlp\.fc1\.":                r"vision_encoder.backbone.layers.\1.mlp.fc1.",
    r"^vision_encoder\.backbone\.blocks\.(\d+)\.mlp\.fc2\.":                r"vision_encoder.backbone.layers.\1.mlp.fc2.",

    # Vision Encoder - FPN Neck
    r"^backbone\.vision_backbone\.neck\.fpn\.(\d+)\.":                      r"vision_encoder.neck.fpn_layers.\1.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.dconv_2x2_0\.":             r"vision_encoder.neck.fpn_layers.\1.scale_layers.0.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.dconv_2x2_1\.":             r"vision_encoder.neck.fpn_layers.\1.scale_layers.2.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.dconv_2x2\.":               r"vision_encoder.neck.fpn_layers.\1.scale_layers.0.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.maxpool_2x2\.":             r"vision_encoder.neck.fpn_layers.\1.scale_layers.0.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.conv_1x1\.":                r"vision_encoder.neck.fpn_layers.\1.proj1.",
    r"^backbone\.vision_backbone\.convs\.(\d+)\.conv_3x3\.":                r"vision_encoder.neck.fpn_layers.\1.proj2.",

    # ============================================================================
    # Text Encoder (CLIP)
    # ============================================================================
    r"^backbone\.language_backbone\.encoder\.":                             r"text_encoder.",
    r"^text_encoder\.token_embedding\.":                                    r"text_encoder.text_model.embeddings.token_embedding.",
    r"^text_encoder\.positional_embedding":                                 r"text_encoder.text_model.embeddings.position_embedding.weight",
    r"^text_encoder\.ln_final\.":                                           r"text_encoder.text_model.final_layer_norm.",
    r"^text_encoder\.text_projection":                                      r"text_encoder.text_projection.weight",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.attn\.in_proj_":       r"text_encoder.text_model.encoder.layers.\1.self_attn.in_proj_",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.attn\.out_proj\.":     r"text_encoder.text_model.encoder.layers.\1.self_attn.out_proj.",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.ln_1\.":               r"text_encoder.text_model.encoder.layers.\1.layer_norm1.",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.ln_2\.":               r"text_encoder.text_model.encoder.layers.\1.layer_norm2.",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.mlp\.c_fc\.":          r"text_encoder.text_model.encoder.layers.\1.mlp.fc1.",
    r"^text_encoder\.transformer\.resblocks\.(\d+)\.mlp\.c_proj\.":        r"text_encoder.text_model.encoder.layers.\1.mlp.fc2.",
    r"^backbone\.language_backbone\.resizer\.":                             r"text_projection.",

    # ============================================================================
    # Geometry Encoder
    # ============================================================================
    r"^geometry_encoder\.encode\.(\d+)\.cross_attn_image\.out_proj\.":     r"geometry_encoder.layers.\1.cross_attn.o_proj.",
    r"^geometry_encoder\.encode\.(\d+)\.cross_attn_image\.":               r"geometry_encoder.layers.\1.cross_attn.",
    r"^geometry_encoder\.encode\.(\d+)\.self_attn\.out_proj\.":            r"geometry_encoder.layers.\1.self_attn.o_proj.",
    r"^geometry_encoder\.encode\.(\d+)\.self_attn\.":                      r"geometry_encoder.layers.\1.self_attn.",
    r"^geometry_encoder\.encode\.(\d+)\.linear1\.":                        r"geometry_encoder.layers.\1.mlp.fc1.",
    r"^geometry_encoder\.encode\.(\d+)\.linear2\.":                        r"geometry_encoder.layers.\1.mlp.fc2.",
    r"^geometry_encoder\.encode\.(\d+)\.norm1\.":                          r"geometry_encoder.layers.\1.layer_norm1.",
    r"^geometry_encoder\.encode\.(\d+)\.norm2\.":                          r"geometry_encoder.layers.\1.layer_norm2.",
    r"^geometry_encoder\.encode\.(\d+)\.norm3\.":                          r"geometry_encoder.layers.\1.layer_norm3.",
    r"^geometry_encoder\.img_pre_norm\.":                                   r"geometry_encoder.vision_layer_norm.",
    r"^geometry_encoder\.norm\.":                                           r"geometry_encoder.prompt_layer_norm.",
    r"^geometry_encoder\.encode_norm\.":                                    r"geometry_encoder.output_layer_norm.",

    # ============================================================================
    # DETR Encoder
    # ============================================================================
    r"^transformer\.encoder\.layers\.(\d+)\.cross_attn_image\.out_proj\.":  r"detr_encoder.layers.\1.cross_attn.o_proj.",
    r"^transformer\.encoder\.layers\.(\d+)\.cross_attn_image\.":            r"detr_encoder.layers.\1.cross_attn.",
    r"^transformer\.encoder\.layers\.(\d+)\.self_attn\.out_proj\.":         r"detr_encoder.layers.\1.self_attn.o_proj.",
    r"^transformer\.encoder\.layers\.(\d+)\.self_attn\.":                   r"detr_encoder.layers.\1.self_attn.",
    r"^transformer\.encoder\.layers\.(\d+)\.cross_attn\.out_proj\.":        r"detr_encoder.layers.\1.cross_attn.o_proj.",
    r"^transformer\.encoder\.layers\.(\d+)\.cross_attn\.":                  r"detr_encoder.layers.\1.cross_attn.",
    r"^transformer\.encoder\.layers\.(\d+)\.linear1\.":                     r"detr_encoder.layers.\1.mlp.fc1.",
    r"^transformer\.encoder\.layers\.(\d+)\.linear2\.":                     r"detr_encoder.layers.\1.mlp.fc2.",
    r"^transformer\.encoder\.layers\.(\d+)\.norm1\.":                       r"detr_encoder.layers.\1.layer_norm1.",
    r"^transformer\.encoder\.layers\.(\d+)\.norm2\.":                       r"detr_encoder.layers.\1.layer_norm2.",
    r"^transformer\.encoder\.layers\.(\d+)\.norm3\.":                       r"detr_encoder.layers.\1.layer_norm3.",

    # ============================================================================
    # DETR Decoder
    # ============================================================================
    r"^transformer\.decoder\.query_embed\.":                                r"detr_decoder.query_embed.",
    r"^transformer\.decoder\.reference_points\.":                           r"detr_decoder.reference_points.",
    r"^transformer\.decoder\.instance_query_embed\.":                       r"detr_decoder.instance_query_embed.",
    r"^transformer\.decoder\.instance_reference_points\.":                  r"detr_decoder.instance_reference_points.",
    r"^transformer\.decoder\.presence_token\.":                             r"detr_decoder.presence_token.",
    r"^transformer\.decoder\.presence_token_head\.layers\.0\.":             r"detr_decoder.presence_head.layer1.",
    r"^transformer\.decoder\.presence_token_head\.layers\.1\.":             r"detr_decoder.presence_head.layer2.",
    r"^transformer\.decoder\.presence_token_head\.layers\.2\.":             r"detr_decoder.presence_head.layer3.",
    r"^transformer\.decoder\.presence_token_out_norm\.":                    r"detr_decoder.presence_layer_norm.",
    r"^transformer\.decoder\.norm\.":                                       r"detr_decoder.output_layer_norm.",
    r"^transformer\.decoder\.bbox_embed\.layers\.0\.":                      r"detr_decoder.box_head.layer1.",
    r"^transformer\.decoder\.bbox_embed\.layers\.1\.":                      r"detr_decoder.box_head.layer2.",
    r"^transformer\.decoder\.bbox_embed\.layers\.2\.":                      r"detr_decoder.box_head.layer3.",
    r"^transformer\.decoder\.instance_bbox_embed\.layers\.0\.":             r"detr_decoder.instance_box_head.layer1.",
    r"^transformer\.decoder\.instance_bbox_embed\.layers\.1\.":             r"detr_decoder.instance_box_head.layer2.",
    r"^transformer\.decoder\.instance_bbox_embed\.layers\.2\.":             r"detr_decoder.instance_box_head.layer3.",
    r"^transformer\.decoder\.ref_point_head\.layers\.0\.":                  r"detr_decoder.ref_point_head.layer1.",
    r"^transformer\.decoder\.ref_point_head\.layers\.1\.":                  r"detr_decoder.ref_point_head.layer2.",
    r"^transformer\.decoder\.boxRPB_embed_x\.layers\.0\.":                  r"detr_decoder.box_rpb_embed_x.layer1.",
    r"^transformer\.decoder\.boxRPB_embed_x\.layers\.1\.":                  r"detr_decoder.box_rpb_embed_x.layer2.",
    r"^transformer\.decoder\.boxRPB_embed_y\.layers\.0\.":                  r"detr_decoder.box_rpb_embed_y.layer1.",
    r"^transformer\.decoder\.boxRPB_embed_y\.layers\.1\.":                  r"detr_decoder.box_rpb_embed_y.layer2.",
    r"^transformer\.decoder\.layers\.(\d+)\.self_attn\.out_proj\.":         r"detr_decoder.layers.\1.self_attn.o_proj.",
    r"^transformer\.decoder\.layers\.(\d+)\.self_attn\.":                   r"detr_decoder.layers.\1.self_attn.",
    r"^transformer\.decoder\.layers\.(\d+)\.ca_text\.out_proj\.":           r"detr_decoder.layers.\1.text_cross_attn.o_proj.",
    r"^transformer\.decoder\.layers\.(\d+)\.ca_text\.":                     r"detr_decoder.layers.\1.text_cross_attn.",
    r"^transformer\.decoder\.layers\.(\d+)\.cross_attn\.out_proj\.":        r"detr_decoder.layers.\1.vision_cross_attn.o_proj.",
    r"^transformer\.decoder\.layers\.(\d+)\.cross_attn\.":                  r"detr_decoder.layers.\1.vision_cross_attn.",
    r"^transformer\.decoder\.layers\.(\d+)\.linear1\.":                     r"detr_decoder.layers.\1.mlp.fc1.",
    r"^transformer\.decoder\.layers\.(\d+)\.linear2\.":                     r"detr_decoder.layers.\1.mlp.fc2.",
    r"^transformer\.decoder\.layers\.(\d+)\.norm1\.":                       r"detr_decoder.layers.\1.vision_cross_attn_layer_norm.",
    r"^transformer\.decoder\.layers\.(\d+)\.catext_norm\.":                 r"detr_decoder.layers.\1.text_cross_attn_layer_norm.",
    r"^transformer\.decoder\.layers\.(\d+)\.norm2\.":                       r"detr_decoder.layers.\1.self_attn_layer_norm.",
    r"^transformer\.decoder\.layers\.(\d+)\.norm3\.":                       r"detr_decoder.layers.\1.mlp_layer_norm.",

    # ============================================================================
    # Dot Product Scoring
    # ============================================================================
    r"^dot_prod_scoring\.prompt_mlp\.layers\.0\.":                          r"dot_product_scoring.text_mlp.layer1.",
    r"^dot_prod_scoring\.prompt_mlp\.layers\.1\.":                          r"dot_product_scoring.text_mlp.layer2.",
    r"^dot_prod_scoring\.prompt_mlp\.out_norm\.":                           r"dot_product_scoring.text_mlp_out_norm.",
    r"^dot_prod_scoring\.prompt_proj\.":                                    r"dot_product_scoring.text_proj.",
    r"^dot_prod_scoring\.hs_proj\.":                                        r"dot_product_scoring.query_proj.",

    # ============================================================================
    # Mask Decoder
    # ============================================================================
    r"^segmentation_head\.pixel_decoder\.conv_layers\.(\d+)\.":             r"mask_decoder.pixel_decoder.conv_layers.\1.",
    r"^segmentation_head\.pixel_decoder\.norms\.(\d+)\.":                   r"mask_decoder.pixel_decoder.norms.\1.",
    r"^segmentation_head\.mask_embed\.layers\.(\d+)\.":                     r"mask_decoder.mask_embedder.layers.\1.",
    r"^segmentation_head\.mask_predictor\.mask_embed\.layers\.(\d+)\.":     r"mask_decoder.mask_embedder.layers.\1.",
    r"^segmentation_head\.instance_seg_head\.":                             r"mask_decoder.instance_projection.",
    r"^segmentation_head\.semantic_seg_head\.":                             r"mask_decoder.semantic_projection.",
    r"^segmentation_head\.cross_attend_prompt\.out_proj\.":                 r"mask_decoder.prompt_cross_attn.o_proj.",
    r"^segmentation_head\.cross_attend_prompt\.":                           r"mask_decoder.prompt_cross_attn.",
    r"^segmentation_head\.cross_attn_norm\.":                               r"mask_decoder.prompt_cross_attn_norm.",
}
# fmt: on


def convert_old_keys_to_new_keys(state_dict_keys: list[str]) -> dict[str, str]:
    """
    Convert original SAM3 checkpoint keys to HuggingFace format.

    This function applies regex patterns to efficiently rename keys in bulk.

    Args:
        state_dict_keys: List of original checkpoint keys

    Returns:
        Dictionary mapping original keys to new keys
    """
    output_dict = {}
    if state_dict_keys is not None:
        old_text = "\n".join(state_dict_keys)
        new_text = old_text

        # Apply all regex patterns
        for pattern, replacement in ORIGINAL_TO_CONVERTED_KEY_MAPPING.items():
            new_text = re.sub(pattern, replacement, new_text, flags=re.MULTILINE)

        output_dict = dict(zip(old_text.split("\n"), new_text.split("\n")))

    return output_dict


def split_qkv(state_dict: dict) -> dict:
    """
    Split combined QKV weights/biases into separate Q, K, V projections.

    Both the vision backbone and text encoder in the original SAM3 use combined QKV projections,
    but the refactored model uses separate Q, K, V projections.

    Args:
        state_dict: State dictionary with combined QKV weights

    Returns:
        State dictionary with split Q, K, V weights
    """
    # Handle vision backbone: .attention.qkv.* → .attention.{q,k,v}_proj.*
    vision_keys_to_split = [key for key in state_dict.keys() if ".attention.qkv." in key]

    for key in vision_keys_to_split:
        qkv = state_dict.pop(key)
        # Split into 3 equal chunks along dimension 0 (output dimension)
        q, k, v = torch.chunk(qkv, 3, dim=0)

        # Create new keys for q_proj, k_proj, v_proj
        state_dict[key.replace(".qkv.", ".q_proj.")] = q
        state_dict[key.replace(".qkv.", ".k_proj.")] = k
        state_dict[key.replace(".qkv.", ".v_proj.")] = v

    # Handle all attention layers with in_proj_* (text encoder, DETR decoder cross-attention, mask decoder)
    # These use: .{attn_type}.in_proj_* → .{attn_type}.{q,k,v}_proj.*
    in_proj_keys_to_split = [key for key in state_dict.keys() if ".in_proj_" in key]

    for key in in_proj_keys_to_split:
        in_proj = state_dict.pop(key)
        # Split into 3 equal chunks along dimension 0 (output dimension)
        q, k, v = torch.chunk(in_proj, 3, dim=0)

        # Create new keys for q_proj, k_proj, v_proj
        # Replace "in_proj_weight" with "q_proj.weight" (or "in_proj_bias" with "q_proj.bias")
        if key.endswith("in_proj_weight"):
            base_key = key.replace("in_proj_weight", "")
            state_dict[base_key + "q_proj.weight"] = q
            state_dict[base_key + "k_proj.weight"] = k
            state_dict[base_key + "v_proj.weight"] = v
        elif key.endswith("in_proj_bias"):
            base_key = key.replace("in_proj_bias", "")
            state_dict[base_key + "q_proj.bias"] = q
            state_dict[base_key + "k_proj.bias"] = k
            state_dict[base_key + "v_proj.bias"] = v

    return state_dict


def load_original_state_dict(checkpoint_path: str) -> dict[str, torch.Tensor]:
    """Load the original SAM3 checkpoint."""
    print(f"Loading original checkpoint from {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # Handle different checkpoint formats
    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    print(f"Loaded {len(state_dict)} keys from checkpoint")
    return state_dict


def get_sam3_config(
    vision_config: dict | None = None,
    text_config: dict | None = None,
) -> Sam3Config:
    """
    Create SAM3 configuration.

    Args:
        vision_config: Optional vision encoder configuration overrides
        text_config: Optional text encoder configuration overrides

    Returns:
        Sam3Config instance
    """
    config = Sam3Config()

    # Update with any provided overrides
    if vision_config is not None:
        for key, value in vision_config.items():
            setattr(config.vision_config, key, value)

    if text_config is not None:
        # Text config is a CLIPTextConfig
        for key, value in text_config.items():
            setattr(config.text_config, key, value)

    return config


def convert_sam3_checkpoint(
    checkpoint_path: str,
    output_path: str,
    config: Sam3Config | None = None,
    push_to_hub: bool = False,
    repo_id: str | None = None,
):
    """
    Convert SAM3 checkpoint from original format to HuggingFace format.

    Args:
        checkpoint_path: Path to the original checkpoint file
        output_path: Path to save the converted checkpoint
        config: Optional Sam3Config to use (otherwise creates default)
        push_to_hub: Whether to push the model to the Hub
        repo_id: Repository ID for pushing to Hub
    """
    # Create output directory
    os.makedirs(output_path, exist_ok=True)

    # Load configuration
    if config is None:
        config = get_sam3_config()

    config.architectures = ["Sam3Model"]
    config.save_pretrained(output_path)
    print("Model config saved successfully")

    # Load and convert weights
    print("Loading original checkpoint...")
    state_dict_old = load_original_state_dict(checkpoint_path)

    print("Converting checkpoint keys...")
    all_keys = list(state_dict_old.keys())
    key_mapping = convert_old_keys_to_new_keys(all_keys)

    # Create new state dict with converted keys
    state_dict_new = {}

    for old_key in all_keys:
        new_key = key_mapping.get(old_key, old_key)
        # Special handling: Strip cls token from vision backbone position embeddings
        if new_key == "vision_encoder.backbone.embeddings.position_embeddings":
            # Original has [1, 577, 1024] with cls token, but refactored expects [1, 576, 1024] without cls token
            # Strip the first position (cls token position)
            state_dict_new[new_key] = state_dict_old[old_key][:, 1:, :]
        else:
            state_dict_new[new_key] = state_dict_old[old_key]

    del state_dict_old
    gc.collect()

    # Split combined QKV projections into separate Q, K, V projections
    print("Splitting QKV projections...")
    state_dict_new = split_qkv(state_dict_new)

    # Transpose CLIP text projection (stored transposed in original)
    if "text_encoder.text_projection.weight" in state_dict_new:
        print("Transposing CLIP text_projection...")
        state_dict_new["text_encoder.text_projection.weight"] = state_dict_new["text_encoder.text_projection.weight"].T

    # Load into HF model
    print("Loading weights into Sam3Model...")
    model = Sam3Model(config)
    missing_keys, unexpected_keys = model.load_state_dict(state_dict_new, strict=False)

    if missing_keys:
        logger.warning(f"Missing keys ({len(missing_keys)}):")
        for key in missing_keys:  # Show more keys for debugging
            logger.warning(f"  - {key}")

    if unexpected_keys:
        logger.warning(f"Unexpected keys ({len(unexpected_keys)}):")
        for key in unexpected_keys:  # Show more keys for debugging
            logger.warning(f"  - {key}")

    # Note: Some missing/unexpected keys are expected:
    # - vision_encoder.backbone.embeddings.patch_embeddings.projection.bias: patch projection has bias=False
    # - geometry_encoder.mask_encoder.projection.*: this is nn.Identity() in original (no weights)
    # - rotary_emb.rope_embeddings: pre-computed in original, computed on-the-fly in refactored
    # - text_encoder.text_projection.bias: projection layer might not have bias

    # Save model
    print(f"Saving converted model to {output_path}")
    model.save_pretrained(
        output_path,
    )

    # Save processor
    print("Creating and saving processor...")
    image_processor = Sam3ImageProcessor()
    tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32", max_length=32, model_max_length=32)
    processor = Sam3Processor(image_processor=image_processor, tokenizer=tokenizer)
    processor.save_pretrained(output_path)

    # Push to hub if requested
    if push_to_hub:
        if repo_id is None:
            raise ValueError("repo_id must be provided when push_to_hub=True")
        print(f"Pushing model to Hub: {repo_id}")
        model.push_to_hub(repo_id)
        processor.push_to_hub(repo_id)

    print("Conversion complete!")
    print(f"Model saved successfully to: {output_path}")

    # Cleanup
    del state_dict_new, model
    gc.collect()

    # Verify the conversion by reloading
    print("\nVerifying converted checkpoint can be loaded...")
    try:
        model = Sam3Model.from_pretrained(output_path)
        param_count = sum(p.numel() for p in model.parameters())
        print(f"✓ Successfully loaded model with {param_count:,} parameters")
        del model
        gc.collect()
    except Exception as e:
        print(f"✗ Failed to reload model: {e}")

    print("\n" + "=" * 80)
    print("Conversion finished!")
    print("=" * 80)
    print(f"Output directory: {output_path}")
    print("\nTo test the model, you can run:")
    print(">>> from transformers import Sam3Model")
    print(f">>> model = Sam3Model.from_pretrained('{output_path}')")
    print("=" * 80)


def main():
    parser = argparse.ArgumentParser(description="Convert SAM3 checkpoint to HuggingFace format")
    parser.add_argument(
        "--checkpoint_path",
        type=str,
        required=True,
        help="Path to the original SAM3 checkpoint file",
    )
    parser.add_argument(
        "--output_path",
        type=str,
        required=True,
        help="Path to save the converted checkpoint",
    )
    parser.add_argument(
        "--push_to_hub",
        action="store_true",
        help="Whether to push the converted model to the Hugging Face Hub",
    )
    parser.add_argument(
        "--repo_id",
        type=str,
        default=None,
        help="Repository ID for pushing to Hub (e.g., 'facebook/sam3-large')",
    )

    args = parser.parse_args()

    convert_sam3_checkpoint(
        checkpoint_path=args.checkpoint_path,
        output_path=args.output_path,
        push_to_hub=args.push_to_hub,
        repo_id=args.repo_id,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sam_infer.py
"""Zero-shot building segmentation with SAM 3.1 (text prompt, no training).

Writes data/<aoi>_prob.npy: per-pixel max over detected instances of
mask probability x instance score, so the site's threshold slider still has
a continuous map to sweep.

facebook/sam3.1 ships only Meta's native checkpoint (gated; request access on
Hugging Face). Meta's own package assumes CUDA, so the detector weights are
remapped in memory onto transformers' Sam3Model, which runs on MPS/CPU.
"""
import os, sys
import numpy as np, torch
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import CLIPTokenizerFast, Sam3Config, Sam3ImageProcessor, Sam3Model, Sam3Processor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from convert_sam3_to_hf import convert_old_keys_to_new_keys, split_qkv

DEV    = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
TILE   = 384     # ~300 m per tile; SAM resizes to 1008, so rooftops get ~2.6x
STEP   = 256
PROMPT = os.environ.get("SAM_PROMPT", "building")

def load_sam31():
    ckpt = torch.load(hf_hub_download("facebook/sam3.1", "sam3.1_multiplex.pt"),
                      map_location="cpu", weights_only=True, mmap=True)
    ckpt = ckpt.get("model", ckpt)
    # the image detector is the "detector." half; the "tracker." half is video-only
    old = {k[len("detector."):]: v for k, v in ckpt.items() if k.startswith("detector.")}
    names = convert_old_keys_to_new_keys(list(old))
    new = {}
    for k, v in old.items():
        nk = names[k]
        new[nk] = v[:, 1:, :] if nk == "vision_encoder.backbone.embeddings.position_embeddings" else v
    new = split_qkv(new)
    if "text_encoder.text_projection.weight" in new:
        new["text_encoder.text_projection.weight"] = new["text_encoder.text_projection.weight"].T
    model = Sam3Model(Sam3Config())
    missing, unexpected = model.load_state_dict(new, strict=False)
    # same allowances as the upstream converter: bias-free projections, the
    # identity mask projection, and RoPE tables computed on the fly. SAM 3.1
    # also drops the 0.5x FPN level, which Sam3Model computes but discards.
    real = [k for k in missing if not k.endswith(("projection.bias", "rope_embeddings"))
            and "mask_encoder.projection" not in k and "neck.fpn_layers.3." not in k]
    assert not real, f"SAM 3.1 weights did not cover: {real[:10]}"
    print(f"  SAM 3.1 detector: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params, "
          f"{len(unexpected)} tracker-only keys skipped", flush=True)
    return model.to(DEV).eval()

model = load_sam31()
proc  = Sam3Processor(image_processor=Sam3ImageProcessor(),
                      tokenizer=CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32",
                                                                  max_length=32, model_max_length=32))

@torch.inference_mode()
def tile_prob(tile):
    inp = proc(images=Image.fromarray(tile), text=PROMPT, return_tensors="pt").to(DEV)
    out = model(**inp)
    score = out.pred_logits[0].sigmoid()
    if out.presence_logits is not None:
        score = score * out.presence_logits[0].sigmoid()
    masks = torch.nn.functional.interpolate(out.pred_masks[0][None], size=tile.shape[:2],
                                            mode="bilinear", align_corners=False)[0].sigmoid()
    return (masks * score.view(-1, 1, 1)).amax(0).float().cpu().numpy()

def infer(img):
    """Overlapping tiles, cosine-feathered, same scheme as the old U-Net export."""
    H, W, _ = img.shape
    acc = np.zeros((H, W), np.float32); wgt = np.zeros((H, W), np.float32)
    ramp = np.maximum(np.hanning(TILE)[:, None] * np.hanning(TILE)[None, :], 1e-3)
    ys = sorted(set(list(range(0, H - TILE + 1, STEP)) + [H - TILE]))
    xs = sorted(set(list(range(0, W - TILE + 1, STEP)) + [W - TILE]))
    for y in ys:
        for x in xs:
            p = tile_prob(img[y:y + TILE, x:x + TILE])
            acc[y:y + TILE, x:x + TILE] += p * ramp; wgt[y:y + TILE, x:x + TILE] += ramp
    return acc / np.maximum(wgt, 1e-6)

def best_iou(prob, truth):
    """Best IoU over the same 0.05-0.95 threshold sweep the site reports."""
    return max(((round(float(t), 2), ((prob >= t) & truth).sum() / max(((prob >= t) | truth).sum(), 1))
                for t in np.arange(0.05, 0.96, 0.05)), key=lambda r: r[1])

if __name__ == "__main__":
    for name in sys.argv[1:] or ["ponce", "guaynabo"]:
        img = np.asarray(Image.open(f"data/{name}_img.jpg").convert("RGB"))
        prob = infer(img)
        np.save(f"data/{name}_prob.npy", prob)
        t, iou = best_iou(prob, np.asarray(Image.open(f"data/{name}_mask.png")) > 127)
        print(f"  {name:9s} prompt={PROMPT!r} best IoU {iou:.3f} @ t={t:.2f}", flush=True)


In [ ]:
%%writefile export_assets.py
import json, math, os
import numpy as np, cv2
from PIL import Image

TEST = ["ponce", "guaynabo"]
OUT  = os.environ.get("GEOAI_OUT", "assets/geoai")
os.makedirs(OUT, exist_ok=True)

def infer(name):
    """SAM 3.1 probability map written by sam_infer.py."""
    return np.load(f"data/{name}_prob.npy")

SHIP=1024
summary={}
for name in TEST:
    truth=(np.asarray(Image.open(f"data/{name}_mask.png"))>127)
    prob=infer(name)
    # ship at 1024
    Image.open(f"data/{name}_img.jpg").convert("RGB").resize((SHIP,SHIP),Image.LANCZOS)\
         .save(f"{OUT}/{name}_img.jpg",quality=82,optimize=True)
    Image.fromarray((prob*255).astype(np.uint8)).resize((SHIP,SHIP),Image.BILINEAR)\
         .save(f"{OUT}/{name}_prob.png",optimize=True)
    Image.fromarray((truth*255).astype(np.uint8)).resize((SHIP,SHIP),Image.NEAREST)\
         .save(f"{OUT}/{name}_truth.png",optimize=True)
    # metrics at full res
    rows=[]
    for t in np.arange(0.05,0.96,0.05):
        pr=prob>=t; tp=int((pr&truth).sum()); fp=int((pr&~truth).sum()); fn=int((~pr&truth).sum())
        prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1); iou=tp/max(tp+fp+fn,1)
        rows.append(dict(t=round(float(t),2),precision=round(prec,4),recall=round(rec,4),iou=round(iou,4)))
    best=max(rows,key=lambda r:r["iou"])
    summary[name]=dict(rows=rows,best=best)
    print(f"  {name:9s} best IoU {best['iou']:.3f} @ t={best['t']:.2f}  P={best['precision']:.3f} R={best['recall']:.3f}")

# ---- hero footprints: Guaynabo, the tile hero_terrain.py is registered to ----
name="guaynabo"
prob=infer(name); thr=summary[name]["best"]["t"]
binm=((prob>=thr)*255).astype(np.uint8)
binm=cv2.morphologyEx(binm,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
# crop to the hero's 650x720 aspect
H,W=binm.shape; cw=int(H*650/720); x0=(W-cw)//2
crop=binm[:,x0:x0+cw]
cnts,_=cv2.findContours(crop,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
paths=[]
for c in cnts:
    if cv2.contourArea(c) < 90: continue
    ap=cv2.approxPolyDP(c, 1.6, True)
    if len(ap) < 3: continue
    pts=[(p[0][0]*650.0/cw, p[0][1]*720.0/H) for p in ap]
    paths.append("M"+" L".join(f"{x:.1f},{y:.1f}" for x,y in pts)+" Z")
paths=sorted(paths,key=len,reverse=True)[:900]
PROMPT=open("data/prompt.txt").read().strip() if os.path.exists("data/prompt.txt") else "building"
json.dump(dict(count=len(paths),source=name,threshold=thr,model="SAM 3.1",prompt=PROMPT),
          open(f"{OUT}/hero-meta.json","w"))
json.dump(dict(count=len(paths),paths=paths), open(f"{OUT}/hero-footprints.json","w"))
print(f"  hero: {len(paths)} footprints from {name}")
json.dump(summary, open(f"{OUT}/metrics.json","w"), indent=1)
for f in sorted(os.listdir(OUT)):
    print(f"   {f:26s} {os.path.getsize(os.path.join(OUT,f))/1024:8.1f} KB")


In [ ]:
!python acquire.py bayamon ponce guaynabo

In [ ]:
import numpy as np
from PIL import Image
import sam_infer as S     # loads SAM 3.1 (~3.5 GB download on first run)

def load(name):
    return (np.asarray(Image.open(f"data/{name}_img.jpg").convert("RGB")),
            np.asarray(Image.open(f"data/{name}_mask.png")) > 127)

# choose the prompt on a training-set AOI only
img, truth = load("bayamon")
scores = {}
for p in ["building", "roof", "house"]:
    S.PROMPT = p
    t, iou = S.best_iou(S.infer(img), truth)
    scores[p] = iou
    print(f"  bayamon  prompt={p!r:11s} best IoU {iou:.3f} @ t={t:.2f}")
S.PROMPT = max(scores, key=scores.get)
open("data/prompt.txt", "w").write(S.PROMPT)
print("chosen:", S.PROMPT)

In [ ]:
for name in ["ponce", "guaynabo"]:
    img, truth = load(name)
    prob = S.infer(img)
    np.save(f"data/{name}_prob.npy", prob)
    t, iou = S.best_iou(prob, truth)
    print(f"  {name:9s} held out, prompt={S.PROMPT!r} best IoU {iou:.3f} @ t={t:.2f}")

In [ ]:
!GEOAI_OUT=out python export_assets.py
!cd out && zip -q ../geoai_assets.zip * && cd .. && ls -la geoai_assets.zip
from google.colab import files
files.download("geoai_assets.zip")